In [1]:
import polars as pl
import duckdb
from pathlib import Path
import plotly.express as px

In [3]:
con = duckdb.connect("trips.duckdb")
con.execute("SET memory_limit = '10GB'")

path = Path.cwd()
if not Path(path, "data").exists(): path = path.parent
path_yellow = path / "data" / "yellow_taxi_unified.parquet"
path_green = path / "data" / "green_taxi_unified.parquet"
path_uber = path / "data" / "uber_unified.parquet"
path_agg_data = path / "data" / "data_reports_monthly.csv"

In [11]:
color_map = {
    "Yellow": "#F2C200",
    "Green":  "#2E8B57",
    "FHV - High Volume": "#7B68EE",
    "FHV": "#7B68EE",
}

In [ ]:
res1 = con.execute(f"""
    SELECT
    CAST(strptime("Month/Year" || '-01', '%Y-%m-%d') AS DATE) AS month_year_date,
    "Avg Hours Per Day Per Vehicle" as m_hours,
    "License Class" as type
    FROM '{path_agg_data}'
    WHERE "License Class" IN ('Yellow', 'Green', 'FHV - High Volume')    
    ORDER BY month_year_date DESC;

""").pl()

fig = px.line(
    res1,
    x="month_year_date",
    y="m_hours",
    color="type",
    title="Active hours per day per vehicle",
    color_discrete_map=color_map,
)

fig.update_xaxes(dtick="M12", tickformat="%Y")

fig.update_traces(
    hovertemplate="Date: %{x|%b %Y}<br>Mean Hours: %{y}<extra>%{fullData.name}</extra>"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Mean Hours",
)

fig.show()

In [ ]:
# Trip-in-progress hours per day per vehicle
res2 = con.execute(f"""
    SELECT
    CAST(strptime("Month/Year" || '-01', '%Y-%m-%d') AS DATE) AS month_year_date,
    try_cast(replace("Unique Vehicles", ',', '') AS BIGINT) AS unique_vehicles,
    "License Class" as type
    FROM '{path_agg_data}'
    WHERE "License Class" IN ('Yellow', 'Green', 'FHV - High Volume')
    ORDER BY month_year_date DESC;

""").pl()

fig = px.line(
    res2,
    x="month_year_date",
    y="unique_vehicles",
    color="type",
    title="Active hours per day per vehicle",
    color_discrete_map=color_map,
)

fig.update_xaxes(dtick="M12", tickformat="%Y")

fig.update_traces(
    hovertemplate="Date: %{x|%b %Y}<br>Unique Vehicles: %{y}<extra>%{fullData.name}</extra>"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Unique Vehicles",
)

fig.show()

In [ ]:
res3 = con.execute(f"""
WITH
daily_fhv AS (
  SELECT
    CAST(pickup_datetime AS DATE) AS day,
    AVG(total_amount) AS daily_avg
  FROM '{path_uber}'
  WHERE CAST(pickup_datetime AS DATE) >= DATE '2021-01-01'
    AND CAST(pickup_datetime AS DATE) <  DATE '2026-01-01'
  GROUP BY 1
),
monthly_fhv AS (
  SELECT
    CAST(date_trunc('month', day) AS DATE) AS month,
    AVG(daily_avg) AS monthly_avg,
    'FHV - High Volume' AS type
  FROM daily_fhv
  GROUP BY 1
),

daily_yellow AS (
  SELECT
    CAST(pickup_datetime AS DATE) AS day,
    AVG(total_amount) AS daily_avg
  FROM '{path_yellow}'
  WHERE CAST(pickup_datetime AS DATE) >= DATE '2021-01-01'
    AND CAST(pickup_datetime AS DATE) <  DATE '2026-01-01'
  GROUP BY 1
),
monthly_yellow AS (
  SELECT
    CAST(date_trunc('month', day) AS DATE) AS month,
    AVG(daily_avg) AS monthly_avg,
    'Yellow' AS type
  FROM daily_yellow
  GROUP BY 1
),

daily_green AS (
  SELECT
    CAST(pickup_datetime AS DATE) AS day,
    AVG(total_amount) AS daily_avg
  FROM '{path_green}'
  WHERE CAST(pickup_datetime AS DATE) >= DATE '2021-01-01'
    AND CAST(pickup_datetime AS DATE) <  DATE '2026-01-01'
  GROUP BY 1
),
monthly_green AS (
  SELECT
    CAST(date_trunc('month', day) AS DATE) AS month,
    AVG(daily_avg) AS monthly_avg,
    'Green' AS type
  FROM daily_green
  GROUP BY 1
)

SELECT * FROM monthly_fhv
WHERE month >= DATE '2021-01-01' AND month < DATE '2026-01-01'
UNION ALL
SELECT * FROM monthly_yellow
WHERE month >= DATE '2021-01-01' AND month < DATE '2025-12-01'
UNION ALL
SELECT * FROM monthly_green
WHERE month >= DATE '2021-01-01' AND month < DATE '2025-12-01'
ORDER BY month, type;
""").pl()

fig = px.line(
    res3,
    x="month",
    y="monthly_avg",
    color="type",
    title="Precio por viaje medio diario (No incluye propinas)",
    color_discrete_map=color_map,
)

fig.update_xaxes(dtick="M12", tickformat="%Y")
fig.update_layout(
    yaxis_tickprefix="$",
    yaxis_tickformat=",",   # 1,234.56 (cambia a ",.0f" si no quieres decimales)
)

fig.update_traces(
    hovertemplate="Date: %{x|%b %Y}<br>Paga media diaria: $%{y:,.2f}<extra>%{fullData.name}</extra>"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Paga media diaria",
)

fig.show()